### Подключение Спарка

In [2]:
import sys
import os

# Добавляем путь к Spark (найди правильный путь)
spark_paths = [
    "/usr/lib/spark/python",
    "/usr/lib/spark/python/lib/py4j-0.10.9.5-src.zip",
    "/usr/lib/spark/python/lib/pyspark.zip",
]

for p in spark_paths:
    if os.path.exists(p):
        sys.path.insert(0, p)
        print(f"Added: {p}")
    else:
        print(f"Not found: {p}")

# Теперь пробуем импортировать
try:
    from pyspark.sql import SparkSession
    print("PySpark imported successfully!")
except ImportError as e:
    print(f"Error: {e}")

Added: /usr/lib/spark/python
Added: /usr/lib/spark/python/lib/py4j-0.10.9.5-src.zip
Added: /usr/lib/spark/python/lib/pyspark.zip
PySpark imported successfully!


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FraudAnalysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("Spark version:", spark.version)

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/lib/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-log4j12-1.7.30.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/26 20:08:03 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/08/26 20:08:09 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
Spark version: 3.3.2


### Функция анализа данных на ошибки

In [4]:
from pyspark.sql.functions import col, isnull, count, when

def analyze_file(df, file_name):
    """
    Анализирует DataFrame и выводит статистику
    """

    print(f"ANALYSIS: {file_name}")

    
    # 1. Общая информация
    print("\n1. SCHEMA:")
    df.printSchema()
    
    # 2. Количество записей
    total = df.count()
    print(f"\n2. Total rows: {total}")
    
    # 3. Пример данных
    print("\n3. Sample data:")
    df.show(5, truncate=False)
    
    # 4. Проверка на нуллы (исключая transaction_id)
    null_columns = [c for c in df.columns if c != "transaction_id"]
    null_counts = df.select([
        count(when(isnull(col(c)), c)).alias(c) for c in null_columns
    ]).collect()[0]
    
    print("\n4. Null values (excluding transaction_id):")
    for col_name, null_count in zip(null_columns, null_counts):
        print(f"   {col_name}: {null_count}")
    
    # 5. Дубликаты
    unique = df.select("transaction_id").distinct().count()
    duplicates = total - unique
    print(f"\n5. Duplicates: {duplicates}")
    
    # 6. Бизнес-правила
    inconsistent1 = df.filter((col("tx_fraud") == 0) & (col("tx_fraud_scenario") != 0)).count()
    inconsistent2 = df.filter((col("tx_fraud") == 1) & (col("tx_fraud_scenario") == 0)).count()
    print(f"\n6. Business rules violations:")
    print(f"   tx_fraud=0 but scenario!=0: {inconsistent1}")
    print(f"   tx_fraud=1 but scenario=0: {inconsistent2}")
    
    # 7. Статистика по сумме транзакций
    print(f"\n7. tx_amount statistics:")
    df.select("tx_amount").describe().show()
    
    # 8. Выбросы через межквартильный размах (IQR)
    print("\n8. Outliers detection (IQR method):")
    
    # Вычисляем квантили
    quantiles = df.approxQuantile("tx_amount", [0.25, 0.75], 0.05)
    if quantiles and len(quantiles) == 2:
        q1, q3 = quantiles[0], quantiles[1]
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
    
        # Считаем выбросы
        outliers_count = df.filter(
            (col("tx_amount") < lower_bound) | (col("tx_amount") > upper_bound)
        ).count()
        outliers_percent = (outliers_count / total) * 100
    
        print(f"   Q1: {q1:.2f}")
        print(f"   Q3: {q3:.2f}")
        print(f"   IQR: {iqr:.2f}")
        print(f"   Lower bound: {lower_bound:.2f}")
        print(f"   Upper bound: {upper_bound:.2f}")
        print(f"   Outliers (IQR): {outliers_count} ({outliers_percent:.2f}%)")
    else:
        print("   Not enough data to compute quantiles.")
    
    # Возвращаем только ключевые метрики
    return {
        "total": total,
        "duplicates": duplicates,
        "inconsistent1": inconsistent1,
        "inconsistent2": inconsistent2,
        "outliers": outliers_count
    }

### Функция чтения данных

In [5]:
# Читаем только один файл
file_name = "2019-08-22.txt"
path = f"s3a://otus-mlops-ilnur-data/{file_name}"

print(f"Reading: {file_name}")

# 1. Читаем как текст
raw = spark.read.text(path)

# 2. Пропускаем первую строку (заголовок с |)
data = raw.rdd.zipWithIndex().filter(lambda x: x[1] > 0).map(lambda x: x[0])

# 3. Преобразуем обратно в DataFrame
from pyspark.sql import Row
df_raw = data.map(lambda row: Row(value=row.value)).toDF()

# 4. Разбиваем по запятой
from pyspark.sql import functions as F

df = df_raw.select(
    F.split(F.col("value"), ",").alias("cols")
).select(
    F.col("cols")[0].cast("int").alias("transaction_id"),
    F.col("cols")[1].alias("tx_datetime"),
    F.col("cols")[2].cast("int").alias("customer_id"),
    F.col("cols")[3].cast("int").alias("terminal_id"),
    F.col("cols")[4].cast("double").alias("tx_amount"),
    F.col("cols")[5].cast("int").alias("tx_time_seconds"),
    F.col("cols")[6].cast("int").alias("tx_time_days"),
    F.col("cols")[7].cast("int").alias("tx_fraud"),
    F.col("cols")[8].cast("int").alias("tx_fraud_scenario")
)

# Проверяем
print(f"Rows: {df.count()}")
df.show(5, truncate=False)

Reading: 2019-08-22.txt


Rows: 46988418
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|0             |2019-08-22 06:51:03|0          |711        |70.91    |24663          |0           |0       |0                |
|1             |2019-08-22 05:10:37|0          |0          |90.55    |18637          |0           |0       |0                |
|2             |2019-08-22 19:05:33|0          |753        |35.38    |68733          |0           |0       |0                |
|3             |2019-08-22 07:21:33|0          |0          |80.41    |26493          |0           |0       |0                |
|4             |2019-08-22 09:06:17|1          |981        |102.83   |32777          |0         

### Вывод результатов

In [6]:
# Анализируем
stats = analyze_file(df, file_name)

# Итоговый отчёт

print(f"SUMMARY FOR {file_name}")

for key, value in stats.items():
    print(f"{key}: {value}")

ANALYSIS: 2019-08-22.txt

1. SCHEMA:
root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: integer (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: integer (nullable = true)
 |-- tx_time_days: integer (nullable = true)
 |-- tx_fraud: integer (nullable = true)
 |-- tx_fraud_scenario: integer (nullable = true)




2. Total rows: 46988418

3. Sample data:
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|0             |2019-08-22 06:51:03|0          |711        |70.91    |24663          |0           |0       |0                |
|1             |2019-08-22 05:10:37|0          |0          |90.55    |18637          |0           |0       |0                |
|2             |2019-08-22 19:05:33|0          |753        |35.38    |68733          |0           |0       |0                |
|3             |2019-08-22 07:21:33|0          |0          |80.41    |26493          |0           |0       |0                |
|4             |2019-08-22 09:06:17|1          |981        |102.83   


4. Null values (excluding transaction_id):
   tx_datetime: 0
   customer_id: 0
   terminal_id: 0
   tx_amount: 0
   tx_time_seconds: 0
   tx_time_days: 0
   tx_fraud: 0
   tx_fraud_scenario: 0



5. Duplicates: 181



6. Business rules violations:
   tx_fraud=0 but scenario!=0: 0
   tx_fraud=1 but scenario=0: 0

7. tx_amount statistics:


+-------+-----------------+
|summary|        tx_amount|
+-------+-----------------+
|  count|         46988418|
|   mean|54.23395999456729|
| stddev|41.25033514383644|
|    min|              0.0|
|    max|          3773.34|
+-------+-----------------+


8. Outliers detection (IQR method):


[Stage 30:==================================================>     (19 + 2) / 21]

   Q1: 22.78
   Q3: 72.43
   IQR: 49.65
   Lower bound: -51.70
   Upper bound: 146.91
   Outliers (IQR): 1309077 (2.79%)
SUMMARY FOR 2019-08-22.txt
total: 46988418
duplicates: 181
inconsistent1: 0
inconsistent2: 0
outliers: 1309077


#### Всего найдено два типа некорректных данных. Посмотрим также, что даты парсятся корректно.

In [7]:
from pyspark.sql.functions import to_timestamp, col

# Проверяем, что даты парсятся корректно
invalid_date = df.filter(to_timestamp(col("tx_datetime"), "yyyy-MM-dd HH:mm:ss").isNull())
print(f"Invalid date format: {invalid_date.count()}")

[Stage 33:=====================================================>  (20 + 1) / 21]

Invalid date format: 100


In [8]:
# Выведем записи с некорректной датой
df.filter(to_timestamp(col("tx_datetime"), "yyyy-MM-dd HH:mm:ss").isNull()).show(10, truncate=False)

[Stage 37:=============================>                            (1 + 1) / 2]

+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|transaction_id|tx_datetime        |customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+--------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|933817        |2019-08-22 24:00:00|597125     |611        |62.83    |86400          |0           |0       |0                |
|1164488       |2019-08-22 24:00:00|743659     |0          |7.25     |86400          |0           |0       |0                |
|1205236       |2019-08-22 24:00:00|769896     |44         |15.1     |86400          |0           |0       |0                |
|1543099       |2019-08-22 24:00:00|985197     |967        |85.18    |86400          |0           |0       |0                |
|1670385       |2019-08-23 24:00:00|66533      |956        |21.39    |172800         |1           |0       |0  

**24:00:00 - некорректный формат даты**

### Функция очистки данных

In [9]:
from pyspark.sql.functions import col

def clean_data(df):
    """
    Очищает DataFrame от выбросов и дубликатов
    """
    # 1. Удаляем дубликаты
    df = df.dropDuplicates(["transaction_id"])
    
    # 2. Удаляем выбросы по tx_amount (IQR)
    quantiles = df.approxQuantile("tx_amount", [0.25, 0.75], 0.05)
    if quantiles and len(quantiles) == 2:
        q1, q3 = quantiles[0], quantiles[1]
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        df = df.filter(
            (col("tx_amount") >= lower_bound) & (col("tx_amount") <= upper_bound)
        )
    
    # 3. Удаляем строки с отрицательной суммой (если есть)
    df = df.filter(col("tx_amount") >= 0)
    
    return df

In [10]:
print(f"CLEANING DATA FOR {file_name}")

df_clean = clean_data(df)

CLEANING DATA FOR 2019-08-22.txt


#### Удалим строки с неккоретными датами

In [11]:
# Удаляем записи с 24:00:00
df_clean_fin = df_clean.filter(~col("tx_datetime").contains("24:00:00"))

In [12]:
print(f"Rows after cleaning: {df_clean_fin.count()}")

[Stage 44:====================================================>   (16 + 1) / 17]

Rows after cleaning: 45670214


### Сохранение датасета в бакете в формате parquet

In [13]:
# Сохраняем очищенный DataFrame в Parquet
output_path = f"s3a://otus-mlops-ilnur-data/cleaned_fin/{file_name.replace('.txt', '.parquet')}"

df_clean_fin.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Saved cleaned data to: {output_path}")

Saved cleaned data to: s3a://otus-mlops-ilnur-data/cleaned_fin/2019-08-22.parquet
